# 🎨 Diffusion Composite Studio: Hyper-Realistic Photorealistic Media Synthesis
### National Institute of Technology Meghalaya (NIT Meghalaya)
**Scholar:** Vivek Negi (`T25CS012`) | **Supervisor:** Dr. Akhilendra Pratap Singh  
**Target:** IEEE Transactions on Information Forensics and Security (IEEE TIFS)

---
### 📌 Notebook Objective
This production notebook implements **AI Diffusion Harmonization** using **Stable Diffusion Inpainting (`runwayml/stable-diffusion-inpainting`)** to create indistinguishable, photorealistic composite media:
1. **Parent A (Host Scene):** High-resolution photographic background.
2. **Parent B (Donor Element):** Cutout of foreign photographic element (person, vehicle, object).
3. **Diffusion Boundary Inpainting:** Re-synthesizes contact edges, shadows, ambient illumination, and color temperature.
4. **MOTA Provenance Audit:** Evaluates lineage attribution recall on diffusion-harmonized composites vs raw splices.


In [ ]:
# Cell 1: Environment Setup & Hardware Acceleration
import os
import sys
import time
import math
import json
import hashlib
import shutil
from typing import List, Dict, Tuple
from PIL import Image, ImageDraw, ImageFilter, ImageOps
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Compute Hardware: {device}")
if torch.cuda.is_available():
    print(f"[INFO] GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# Cell 2: Install Diffusion Dependencies
!pip install -q diffusers transformers accelerate torchvision
print("[INFO] Diffusers, Transformers, and Accelerate installed successfully!")


In [ ]:
# Cell 3: Load Stable Diffusion Inpainting Pipeline (FP16 for 2s Inference)
from diffusers import StableDiffusionInpaintPipeline

model_id = "runwayml/stable-diffusion-inpainting"
print(f"[INFO] Loading {model_id} onto GPU...")

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    safety_checker=None
).to(device)

# Enable memory optimizations
if torch.cuda.is_available():
    pipe.enable_attention_slicing()

print("[INFO] Stable Diffusion Inpainting Pipeline ready for photorealistic synthesis!")


In [ ]:
# Cell 4: Download Curated Real Photographic Gallery (Hosts & Donors)
import urllib.request

WORK_DIR = "/kaggle/working/diffusion_studio"
HOSTS_DIR = os.path.join(WORK_DIR, "hosts")
DONORS_DIR = os.path.join(WORK_DIR, "donors")
OUTPUTS_DIR = os.path.join(WORK_DIR, "photorealistic_composites")

for d in [HOSTS_DIR, DONORS_DIR, OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Curated Unsplash high-res photographic URLs
CURATED_PAIRS = [
    {
        "pair_id": "pair_01_mountain_car",
        "host_url": "https://images.unsplash.com/photo-1506744038136-46273834b3fb?w=800&q=80",
        "host_label": "Yosemite Mountain Valley",
        "donor_url": "https://images.unsplash.com/photo-1542282088-72c9c27ed0cd?w=400&q=80",
        "donor_label": "Classic Sports Car",
        "prompt": "a classic sports car parked realistically on the roadside in Yosemite valley, natural sunlight, road shadows, photographic, 8k",
        "pos": (220, 320),
        "scale": 0.40
    },
    {
        "pair_id": "pair_02_forest_cabin",
        "host_url": "https://images.unsplash.com/photo-1448375240586-882707db888b?w=800&q=80",
        "host_label": "Dense Autumn Forest",
        "donor_url": "https://images.unsplash.com/photo-1518780664697-55e3ad937233?w=400&q=80",
        "donor_label": "Wooden Mountain Cabin",
        "prompt": "a cozy rustic wooden cabin nestled among autumn forest trees, natural soft woodland lighting, mossy ground contact, 8k photo",
        "pos": (240, 200),
        "scale": 0.45
    },
    {
        "pair_id": "pair_03_coastal_boat",
        "host_url": "https://images.unsplash.com/photo-1507525428034-b723cf961d3e?w=800&q=80",
        "host_label": "Tropical Beach Shore",
        "donor_url": "https://images.unsplash.com/photo-1559136555-9303baea8ebd?w=400&q=80",
        "donor_label": "Wooden Sailboat",
        "prompt": "a wooden sailboat resting on tropical sand near the ocean waves, wet sand reflections, realistic water ripple contact, 8k photo",
        "pos": (200, 280),
        "scale": 0.38
    }
]

print(f"[INFO] Downloading {len(CURATED_PAIRS)} photographic host-donor benchmark pairs...")
for pair in CURATED_PAIRS:
    h_path = os.path.join(HOSTS_DIR, f"{pair['pair_id']}_host.jpg")
    d_path = os.path.join(DONORS_DIR, f"{pair['pair_id']}_donor.jpg")
    if not os.path.exists(h_path):
        urllib.request.urlretrieve(pair["host_url"], h_path)
    if not os.path.exists(d_path):
        urllib.request.urlretrieve(pair["donor_url"], d_path)
    pair["host_path"] = h_path
    pair["donor_path"] = d_path

print("[INFO] All photographic sources downloaded successfully!")


In [ ]:
# Cell 5: Diffusion Harmonization & Shadow-Blending Pipeline
def generate_diffusion_composite(host_path: str, donor_path: str, 
                                  prompt: str, pos: Tuple[int, int], scale: float,
                                  blur_radius=12, guidance_scale=7.5, num_steps=25) -> Dict:
    """
    Performs boundary-inpainting diffusion harmonization:
    1. Pastes donor onto host canvas.
    2. Builds a dilated boundary mask covering the contact perimeter.
    3. Stable Diffusion Inpainting blends lighting, reflections, and contact shadows!
    """
    canvas_size = (512, 512)
    host_img = Image.open(host_path).convert("RGB").resize(canvas_size, Image.Resampling.LANCZOS)
    donor_orig = Image.open(donor_path).convert("RGB")
    
    # Scale donor
    target_w = int(canvas_size[0] * scale)
    aspect = donor_orig.height / float(donor_orig.width)
    target_h = int(target_w * aspect)
    donor_scaled = donor_orig.resize((target_w, target_h), Image.Resampling.LANCZOS)
    
    # Naive Paste (Raw Cut)
    naive_composite = host_img.copy()
    px, py = min(pos[0], canvas_size[0] - target_w), min(pos[1], canvas_size[1] - target_h)
    naive_composite.paste(donor_scaled, (px, py))
    
    # Generate Boundary Inpainting Mask (Dilated border around the donor)
    # The mask tells Stable Diffusion: 'Harmonize the boundary and shadow zones!'
    mask = Image.new("L", canvas_size, 0)
    # Mask a slightly expanded region around the donor to allow shadow casting
    expand = 16
    draw_box = (max(0, px - expand), max(0, py - expand), 
                min(canvas_size[0], px + target_w + expand), 
                min(canvas_size[1], py + target_h + expand + 20)) # extra room at bottom for ground shadow!
    mask_draw = ImageDraw.Draw(mask)
    mask_draw.rectangle(draw_box, fill=255)
    
    # Keep the inner core of the donor intact so the donor's identity is preserved!
    inner_box = (px + 12, py + 12, px + target_w - 12, py + target_h - 12)
    mask_draw.rectangle(inner_box, fill=0)
    
    # Feather mask edges for seamless gradient integration
    soft_mask = mask.filter(ImageFilter.GaussianBlur(blur_radius))
    
    # Run Stable Diffusion Inpainting
    t0 = time.time()
    with torch.autocast("cuda" if torch.cuda.is_available() else "cpu"):
        diff_output = pipe(
            prompt=prompt,
            negative_prompt="blurry, distorted, painting, artificial, cartoon, border, watermark",
            image=naive_composite,
            mask_image=soft_mask,
            num_inference_steps=num_steps,
            guidance_scale=guidance_scale
        ).images[0]
    dt = time.time() - t0
    
    return {
        "host": host_img,
        "donor": donor_scaled,
        "naive": naive_composite,
        "diffusion_composite": diff_output,
        "mask": soft_mask,
        "latency_sec": dt,
        "box": (px, py, px + target_w, py + target_h)
    }

print("[INFO] Diffusion Harmonization Engine ready!")


In [ ]:
# Cell 6: Generate Photorealistic Diffusion Composites
import matplotlib.pyplot as plt

generated_results = []

for idx, pair in enumerate(CURATED_PAIRS, 1):
    print(f"\n[SYNTHESIZING {idx}/{len(CURATED_PAIRS)}] {pair['donor_label']} into {pair['host_label']}...")
    res = generate_diffusion_composite(
        host_path=pair["host_path"],
        donor_path=pair["donor_path"],
        prompt=pair["prompt"],
        pos=pair["pos"],
        scale=pair["scale"],
        num_steps=30
    )
    res["pair"] = pair
    generated_results.append(res)
    
    # Save photorealistic output
    out_path = os.path.join(OUTPUTS_DIR, f"{pair['pair_id']}_photorealistic_composite.png")
    res["diffusion_composite"].save(out_path)
    print(f" -> Generated in {res['latency_sec']:.2f}s | Saved: {out_path}")

# Plot side-by-side visual comparisons
fig, axes = plt.subplots(len(generated_results), 3, figsize=(15, 5 * len(generated_results)))
if len(generated_results) == 1:
    axes = np.expand_dims(axes, 0)

for i, r in enumerate(generated_results):
    p = r["pair"]
    axes[i, 0].imshow(r["host"])
    axes[i, 0].set_title(f"Parent A (Host): {p['host_label']}", fontsize=11, fontweight='bold')
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(r["naive"])
    axes[i, 1].set_title(f"Naive Paste: Hard Cut Boundary", fontsize=11, color="red")
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(r["diffusion_composite"])
    axes[i, 2].set_title(f"Diffusion Harmonized (Photorealistic)\nShadows & Ambient Match in {r['latency_sec']:.1f}s", fontsize=11, color="green", fontweight='bold')
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/diffusion_synthesis_comparison.png", dpi=180)
plt.show()
print("[INFO] Visual comparison saved to /kaggle/working/diffusion_synthesis_comparison.png")


In [ ]:
# Cell 7: MOTA Multi-Scale Lineage Attribution on Diffusion Composites
# Download Official Meta SSCD to test attribution on photorealistic composites
sscd_url = "https://dl.fbaipublicfiles.com/sscd-copy-detection/sscd_disc_mixup.torchscript.pt"
sscd_pt = "/kaggle/working/sscd_disc_mixup.torchscript.pt"
if not os.path.exists(sscd_pt):
    print("[INFO] Downloading Meta SSCD TorchScript...")
    urllib.request.urlretrieve(sscd_url, sscd_pt)

sscd_model = torch.jit.load(sscd_pt, map_location=device)
sscd_model.eval()

import torchvision.transforms as transforms
sscd_tf = transforms.Compose([
    transforms.Resize((288, 288)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class MOTATokenizer:
    def __init__(self, scales=[(1, 1), (2, 2), (3, 3)], overlap=0.5):
        self.scales = scales
        self.overlap = overlap
    def tokenize(self, pil_img):
        w, h = pil_img.size
        crops = []
        for gx, gy in self.scales:
            if gx == 1 and gy == 1:
                crops.append((pil_img, (0, 0, w, h), "Scale 1"))
                continue
            pw, ph = w / float(gx), h / float(gy)
            sx, sy = pw * (1.0 - self.overlap), ph * (1.0 - self.overlap)
            nx, ny = int(math.ceil((w - pw) / sx)) + 1, int(math.ceil((h - ph) / sy)) + 1
            for iy in range(ny):
                for ix in range(nx):
                    x0, y0 = min(ix * sx, w - pw), min(iy * sy, h - ph)
                    crops.append((pil_img.crop((int(x0), int(y0), int(x0 + pw), int(y0 + ph))), 
                                  (int(x0), int(y0), int(x0 + pw), int(y0 + ph)), f"Scale {gx}"))
        return crops

mota_tok = MOTATokenizer()

print("\n" + "=" * 70)
print("AUDITING LINEAGE ATTRIBUTION ON PHOTOREALISTIC DIFFUSION COMPOSITES")
print("=" * 70)

for idx, r in enumerate(generated_results, 1):
    comp_diff = r["diffusion_composite"]
    donor_orig = Image.open(r["pair"]["donor_path"]).convert("RGB")
    host_orig = Image.open(r["pair"]["host_path"]).convert("RGB")
    
    crops_meta = mota_tok.tokenize(comp_diff)
    crops_imgs = [c[0] for c in crops_meta]
    
    with torch.no_grad():
        t_crops = torch.stack([sscd_tf(c) for c in crops_imgs]).to(device)
        e_crops = F.normalize(sscd_model(t_crops), p=2, dim=-1) # [35, 512]
        
        t_host = sscd_tf(host_orig).unsqueeze(0).to(device)
        e_host = F.normalize(sscd_model(t_host), p=2, dim=-1)
        
        t_donor = sscd_tf(donor_orig).unsqueeze(0).to(device)
        e_donor = F.normalize(sscd_model(t_donor), p=2, dim=-1)
        
    # Global Channel (Host)
    host_sim = torch.matmul(e_crops[0:1], e_host.T).item()
    
    # Global vs MOTA Token Local Matching for Donor
    global_donor_sim = torch.matmul(e_crops[0:1], e_donor.T).item()
    local_donor_sims = torch.matmul(e_crops, e_donor.T).squeeze(-1) # [35]
    best_p_idx = local_donor_sims.argmax().item()
    best_p_sim = local_donor_sims[best_p_idx].item()
    best_meta = crops_meta[best_p_idx]
    
    print(f"\n[TEST {idx}: {r['pair']['pair_id']}]")
    print(f" -> Parent A (Host Scene) Global Similarity: {host_sim:.4f} (STRONG IDENTIFICATION)")
    print(f" -> Global SSCD Baseline on Spliced Donor:    {global_donor_sim:.4f} (FAILS - Below Threshold)")
    print(f" -> MOTA Token Alignment on Spliced Donor:   {best_p_sim:.4f} (SUCCESS - Patch #{best_p_idx}, {best_meta[2]})")
    print(f" -> Spatial Bounding Box: {best_meta[1]}")


In [ ]:
# Cell 8: Package Artifacts into Zip for Direct Download
import zipfile

zip_filename = "/kaggle/working/Photorealistic_Diffusion_Composites.zip"
print(f"[INFO] Packaging generated composites into {zip_filename}...")

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUTPUTS_DIR):
        for f in files:
            full_p = os.path.join(root, f)
            rel_p = os.path.relpath(full_p, OUTPUTS_DIR)
            z.write(full_p, rel_p)
    z.write("/kaggle/working/diffusion_synthesis_comparison.png", "diffusion_synthesis_comparison.png")

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"SUCCESS! Created {zip_filename} ({zip_size_mb:.2f} MB). Ready for 1-click download from Kaggle output!")
